# Word Count Predictions
This notebook converts each review description into a word count vector using CountVectorizer

The length of these vectors is equal to the number of unique words in the dataset

Two models, one Naive Bayes and one Neural Network, are trained on this data to predict the wine variety

## Filter Data

In [1]:
import pandas as pd

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'country', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

df[0:10]

,description,variety
3,"Pineapple rind, lemon pith and orange blossom ...",Riesling
4,"Much like the regular bottling from 2012, this...",Pinot Noir
10,"Soft, supple plum envelopes an oaky structure ...",Cabernet Sauvignon
12,"Slightly reduced, this wine offers a chalky, t...",Cabernet Sauvignon
14,Building on 150 years and six generations of w...,Chardonnay
15,Zesty orange peels and apple notes abound in t...,Riesling
20,Ripe aromas of dark berries mingle with ample ...,Red Blend
21,"A sleek mix of tart berry, stem and herb, alon...",Pinot Noir
23,This wine from the Geneseo district offers aro...,Merlot
25,Oak and earth intermingle around robust aromas...,Pinot Noir


## Generate word count vectors and encode labels

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Encode features
#            TfidfVectorizer
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 1), max_df=0.75, binary=False)

# Fit the CountVectorizer to the descriptions
vectorizer.fit(df['description'])

# Transform the descriptions into a sparse matrix of token counts
X = vectorizer.transform(df['description'])

## Encode labels
le = LabelEncoder()
y = le.fit_transform(df['variety'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# Create the Naive Bayes model
nb = MultinomialNB()
# nb = ComplementNB()

# Fit the model to the training data
nb.fit(X_train, y_train)

# Print the accuracy scores
print('Training accuracy: ', nb.score(X_train, y_train))
print('Testing accuracy: ', nb.score(X_test, y_test))

Training accuracy:  0.7872127872127872
Testing accuracy:  0.7452506133894147


## Feedforward Neural Network

In [10]:
from keras.models import Sequential
from keras.layers import Dense

# Create the neural network model
model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(len(varieties), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
1784/1784 [==============================] - 6s 3ms/step - loss: 0.8387 - accuracy: 0.7274 - val_loss: 0.5968 - val_accuracy: 0.8046
Epoch 2/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.4645 - accuracy: 0.8443 - val_loss: 0.5889 - val_accuracy: 0.8042
Epoch 3/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.3594 - accuracy: 0.8811 - val_loss: 0.6138 - val_accuracy: 0.8051
Epoch 4/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.2918 - accuracy: 0.9054 - val_loss: 0.6569 - val_accuracy: 0.7968
Epoch 5/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.2427 - accuracy: 0.9222 - val_loss: 0.7197 - val_accuracy: 0.7933
Epoch 6/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.2033 - accuracy: 0.9346 - val_loss: 0.7840 - val_accuracy: 0.7915
Epoch 7/10
1784/1784 [==============================] - 5s 3ms/step - loss: 0.1740 - accuracy: 0.9440 - val_loss: 0.8562 - val_accuracy: